# Module 3 Lab 3: Textbook Problem 2.2

**CS 82A - Introduction to Data Science**

**Ramisha Tasfia**

**GitHub:** https://github.com/ramishatasfia5156-commits/cs82a-coursework/blob/main/module3/module3_lab3.ipynb

In [1]:
import pandas as pd

df = pd.read_csv('bridges.csv')

print('Shape:', df.shape)
print()
print('Missing values per column:')
print(df.isna().sum())

Shape: (108, 13)

Missing values per column:
Id           0
river        0
location     1
erected      0
purpose      0
length      27
lanes       16
clear-g      2
t-or-d       6
material     2
span        16
rel-l        5
type         2
dtype: int64


## Plausibility Assumption

Length is the only numeric field that describes each bridge. It will be assumed to be in feet. Since all bridges in this dataset cross one of Pittsburgh’s rivers (A = Allegheny, M = Monongahela, O = Ohio, Y = Youghiogheny), we can first write down our two rules before checking any flagged rows:

1. **Plausible range (impossible?):** Any river crossing must be at least long enough to span the river; therefore, any length less than 500 ft would likely be impossible. Also, we are assuming no single crossing could be longer than 5,000 ft (just under 1 mile). Therefore, we treat values outside 500 to 5,000 ft as errors.
2. **IQR fence (extreme?):** Values that fall above Q3 + 1.5 × IQR or below Q1 − 1.5 × IQR would be considered unusual for this dataset. We check these values against the other attributes of the bridge before determining what should be done.

A value outside rule 1 gets corrected or removed. A value outside only rule 2 is kept unless its other attributes contradict it.

## Summary Statistics

In [2]:
df['length'].describe().to_frame().T.round(1)

,count,mean,std,min,25%,50%,75%,max
length,81.0,1567.5,747.5,804.0,1000.0,1300.0,2000.0,4558.0


## Flagged Rows

In [3]:
LOW, HIGH = 500, 5000
s = df['length']
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Plausible range: [{LOW}, {HIGH}]   IQR fences: [{low_fence:.0f}, {high_fence:.0f}]')

outside_range = (s < LOW) | (s > HIGH)
outside_fence = (s < low_fence) | (s > high_fence)
print('Outside plausible range:', outside_range.sum())
print('Outside IQR fence:      ', outside_fence.sum())

cols = ['Id', 'river', 'erected', 'purpose', 'length', 'lanes', 'material', 'span', 'type']
df.loc[outside_range | outside_fence, cols]

Plausible range: [500, 5000]   IQR fences: [-500, 3500]
Outside plausible range: 0
Outside IQR fence:       3


,Id,river,erected,purpose,length,lanes,material,span,type
31,E34,O,1888,RR,4558.0,2.0,STEEL,LONG,SIMPLE-T
44,E46,A,1897,RR,4000.0,2.0,STEEL,LONG,SIMPLE-T
104,E91,O,1975,HIGHWAY,3756.0,6.0,STEEL,LONG,ARCH


In [4]:
# Context: how do the flagged bridges compare to other LONG-span steel bridges?
long_steel = df[(df['span'] == 'LONG') & (df['material'] == 'STEEL')]['length']
print('LONG-span steel bridges:', long_steel.count())
print('Median length:', long_steel.median())
print('Longest unflagged:', long_steel[long_steel <= high_fence].max())

LONG-span steel bridges: 28
Median length: 1671.0
Longest unflagged: 2822.0


## Handling the Flags

There is no length that falls outside the assumed minimum and maximum values for length (500 to 5,000 ft). Therefore, there is no value in this column that would be considered impossible under this assumption. Three bridges have lengths greater than the IQR fence of 3,500 ft:

**E34 (4,558 ft), E46 (4,000 ft), E91 (3,756 ft) → extreme, not errors. Action: kept.**

All three are steel bridges with a LONG span, and their other attributes support their unusually long lengths. E34 and E91 both cross the Ohio River, which forms where the Allegheny and Monongahela meet. Additionally, E91 is a six-lane highway arch bridge from 1975, making it one of the newest and largest bridges represented in the data. Nothing in the other attributes of these bridges contradicts their lengths; therefore, removing them would remove real information.

The effect these extreme values may have on pulling up the mean length is handled through smoothing (below) rather than deletion.

**Noise found outside length:** E107's type is NIL, which is a placeholder rather than an actual bridge type. Therefore, it is treated as missing.

## Missing Values (Textbook Part A)

In [5]:
# E100 and E109 are missing length, lanes, t-or-d, material, span and type:
# every attribute used in Part C except purpose. Too little left to impute from.
print(df.loc[df['Id'].isin(['E100', 'E109']), cols])
df = df[~df['Id'].isin(['E100', 'E109'])].copy()

# NIL is a placeholder, not a type
df['type'] = df['type'].replace('NIL', pd.NA)

# length: median of bridges with the same purpose (overall median if the group has none)
group_median = df.groupby('purpose')['length'].transform('median')
df['length'] = df['length'].fillna(group_median).fillna(df['length'].median())

# lanes: most common lane count for the same purpose
group_mode = df.groupby('purpose')['lanes'].transform(lambda x: x.mode().iloc[0] if x.notna().any() else 2)
df['lanes'] = df['lanes'].fillna(group_mode).astype(int)

# categorical columns: explicit UNKNOWN instead of a guessed category
for col in ['location', 'clear-g', 't-or-d', 'material', 'span', 'rel-l', 'type']:
    df[col] = df[col].astype(object).where(df[col].notna(), 'UNKNOWN')

print()
print('Shape:', df.shape)
print('Missing values remaining:', df.isna().sum().sum())

       Id river  erected  purpose  length  lanes material span type
106  E100     O     1982  HIGHWAY     NaN    NaN      NaN  NaN  NaN
107  E109     A     1986  HIGHWAY     NaN    NaN      NaN  NaN  NaN

Shape: (106, 13)
Missing values remaining: 0


## Smoothing (Textbook Part B)

In [6]:
# Smoothing by bin means: 3 equal-frequency bins, each value replaced by its bin's mean
df['length_bin'] = pd.qcut(df['length'], q=3, labels=['short', 'medium', 'long'])
df['length_smoothed'] = df.groupby('length_bin', observed=True)['length'].transform('mean').round(0)

df.groupby('length_bin', observed=True)['length'].agg(['count', 'min', 'max', 'mean']).round(0)

,count,min,max,mean
length_bin,,,,
short,40,804.0,1200.0,1021.0
medium,32,1240.0,1460.0,1338.0
long,34,1500.0,4558.0,2241.0


## Relationships (Textbook Part C)

In [7]:
print('i. Length vs. purpose')
pd.crosstab(df['purpose'], df['length_bin'])

i. Length vs. purpose


length_bin,short,medium,long
purpose,,,
AQUEDUCT,4,0,0
HIGHWAY,28,18,23
RR,8,13,11
WALK,0,1,0


In [8]:
print('ii. Lanes vs. material')
pd.crosstab(df['material'], df['lanes'])

ii. Lanes vs. material


lanes,1,2,4,6
material,,,,
IRON,1,10,0,0
STEEL,0,53,22,4
WOOD,3,12,1,0


In [9]:
print('iii. Span vs. lanes')
pd.crosstab(df['span'], df['lanes'])

iii. Span vs. lanes


lanes,1,2,4,6
span,,,,
LONG,0,17,10,3
MEDIUM,0,40,12,1
SHORT,2,6,1,0
UNKNOWN,2,12,0,0


## Cleaning Log

| Change | Before | After | Reason |
|--------|--------|-------|--------|
| Screen `length` | 81 values | 0 outside 500–5,000 ft; 3 outside IQR fence | Assumption stated before screening |
| E34, E46, E91 length | 4,558 / 4,000 / 3,756 | Kept | Extreme but consistent with LONG-span steel bridges |
| E100, E109 | 108 rows | 106 rows | Missing every attribute used in Part C |
| E107 type | NIL | UNKNOWN | Placeholder, not a bridge type |
| length missing | 25 NaN | 0 | Median length for the same purpose |
| lanes missing | 14 NaN | 0 | Most common lane count for the same purpose |
| Categorical missing | NaN | UNKNOWN | Avoids inventing categories that would distort Part C |
| length_bin, length_smoothed | — | 2 new columns | Smoothing by bin means (Part B), bins used in Part C |

## AI Disclosure

**AI Disclosure:** Claude (Anthropic) was used to help explain concepts and draft initial code for this lab. All code was reviewed, retyped, and understood by me. I can explain why the plausibility rule is stated before the screening code, how the range rule and IQR fence flag long bridges, why the three flagged bridges were kept, and how smoothing by bin means and `pd.crosstab` prepare the data for Part C.